# Arizona — ARS Title 20 (Insurance) → `data/arizona/ins_codes/*.md`

**Title 20** is Arizona’s insurance statutes ([ARS Detail — Title 20](https://www.azleg.gov/arsDetail/?title=20)). That page lists every section with a `viewdocument` link; each statute’s **plain HTML** lives at **`https://www.azleg.gov/ars/20/…htm`** (no JavaScript required).

This notebook:

1. Downloads **`arsDetail/?title=20`** and parses each section’s **direct `.htm` URL** from the `docName` query parameter.
2. Fetches each **`.htm`** file, extracts text from `<body>`, and saves **`ARS_sec_<section>.md`** under **`data/arizona/ins_codes/`**.

**Volume:** about **1,500** sections — full run takes several minutes. Use **`MAX_SECTIONS`** in the config cell to cap downloads while testing.

**Politeness:** **`REQUEST_DELAY_SEC`** (default **0.15**) between section requests.

Then run **`python -m app.ingest`** from the project root.

In [1]:
%pip install -q httpx beautifulsoup4

You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import parse_qs, unquote, urlparse

import httpx
from bs4 import BeautifulSoup

ARS_TITLE_PAGE = "https://www.azleg.gov/arsDetail/?title=20"
OUT_DIR = Path("data") / "arizona" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-AZ-ARS-20/1.0 (public ARS; educational indexing)"
REQUEST_DELAY_SEC = 0.15
TIMEOUT = 45.0

# 0 = download all Title 20 sections; set e.g. 25 to smoke-test.
MAX_SECTIONS = 0

# Skip re-download if the .md already exists and is non-trivial.
SKIP_EXISTING = True

In [3]:
def parse_title20_index(html: str) -> list[tuple[str, str]]:
    """Return [(section_label, statute_htm_url), ...] from arsDetail page."""
    soup = BeautifulSoup(html, "html.parser")
    out: list[tuple[str, str]] = []
    seen: set[str] = set()
    for a in soup.find_all("a", href=True):
        label = a.get_text(strip=True)
        if not re.match(r"^20-\d", label):
            continue
        href = a["href"]
        if "docName=" not in href:
            continue
        abs_h = "https://www.azleg.gov" + href if href.startswith("/") else href
        q = parse_qs(urlparse(abs_h).query)
        doc = unquote(q.get("docName", [""])[0])
        if not doc.startswith("http") or "/ars/20/" not in doc:
            continue
        if label in seen:
            continue
        seen.add(label)
        out.append((label, doc))
    return out


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9A-Za-z.-]+", "_", label.replace(".", "_"))
    return f"ARS_sec_{safe}.md"


def extract_statute_body(html: str) -> tuple[str, str]:
    """Return (title_from_HEAD, plain_text from BODY)."""
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    body = soup.find("body")
    text = body.get_text("\n", strip=True) if body else soup.get_text("\n", strip=True)
    return title_txt, text


def download_title20() -> dict[str, int]:
    with httpx.Client(
        headers={"User-Agent": USER_AGENT},
        timeout=TIMEOUT,
        follow_redirects=True,
        http2=False,
    ) as client:
        r = client.get(ARS_TITLE_PAGE)
        r.raise_for_status()
        time.sleep(REQUEST_DELAY_SEC)
        all_pairs = parse_title20_index(r.text)
        print(f"Found {len(all_pairs)} sections on {ARS_TITLE_PAGE}")
        (OUT_DIR / "_arizona_title20_urls.txt").write_text(
            "\n".join(f"{lab}\t{u}" for lab, u in all_pairs),
            encoding="utf-8",
        )
        pairs = all_pairs if not MAX_SECTIONS else all_pairs[:MAX_SECTIONS]
        if MAX_SECTIONS:
            print(f"Limited downloads to first {len(pairs)} sections (MAX_SECTIONS)")

        wrote, skipped, failed = 0, 0, 0
        for i, (label, statute_url) in enumerate(pairs, 1):
            dest = OUT_DIR / label_to_filename(label)
            if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
                skipped += 1
                continue
            try:
                sr = client.get(statute_url)
                sr.raise_for_status()
                time.sleep(REQUEST_DELAY_SEC)
                head_t, body_t = extract_statute_body(sr.text)
                title = head_t or f"A.R.S. § {label}"
                md = (
                    f"# {title}\n\n"
                    f"**Arizona Revised Statutes — Title 20 (Insurance)**\n\n"
                    f"**Official source:** {statute_url}\n\n"
                    f"**Section:** {label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
            if i % 200 == 0:
                print(f"… {i}/{len(pairs)} (wrote={wrote} skipped={skipped} failed={failed})")

    (OUT_DIR / "_arizona_title20_urls.txt").write_text(
        "\n".join(f"{lab}\t{u}" for lab, u in pairs),
        encoding="utf-8",
    )
    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title20()

Found 1539 sections on https://www.azleg.gov/arsDetail/?title=20
… 200/1539 (wrote=200 skipped=0 failed=0)
… 400/1539 (wrote=400 skipped=0 failed=0)
… 600/1539 (wrote=600 skipped=0 failed=0)
… 800/1539 (wrote=800 skipped=0 failed=0)
… 1000/1539 (wrote=1000 skipped=0 failed=0)
… 1200/1539 (wrote=1200 skipped=0 failed=0)
… 1400/1539 (wrote=1400 skipped=0 failed=0)
Done. wrote=1539 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/arizona/ins_codes


{'wrote': 1539, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the repository root.